# Stage1 保存目标与同日风险上限语义检查

原投影空位计划在结果前断言失败，本诊断只读确切失败条件和日期。

In [1]:
import hashlib,json,os
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_original_0bp_projection_dollar_slack_20261023';plan=json.loads((card/'plan.json').read_text());diag=json.loads((card/'semantic_diagnostic_plan.json').read_text());src=plan['source'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert diag['frozen'] and not (card/'semantic_diagnostic_result.json').exists()
base=root/src['stage1_card']/'qlib_recorder'/src['stage1_experiment_id']/src['stage1_recorder_id']/'artifacts';assert sha(base/'inputs/frame')==src['stage1_frame_sha256'] and sha(base/'paths/A_active_research_replay_0')==src['stage1_path_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1';qlib.init(provider_uri=str(root/src['provider_uri']),region='us')
with R.uri_context(str(root/src['stage1_card']/'qlib_recorder')):
    rec=R.get_recorder(recorder_id=src['stage1_recorder_id'],experiment_id=src['stage1_experiment_id']);frame=rec.load_object('inputs/frame').reset_index(drop=True);path=rec.load_object('paths/A_active_research_replay_0')
replay=path['replay'].reset_index(drop=True);target=path['targets'].reset_index(drop=True)
assert len(frame)==len(replay)==len(target)==944 and pd.to_datetime(frame.signal_date).equals(pd.to_datetime(target.signal_date))
rawrisk=frame.soxl_weight.to_numpy(float)+frame.soxx_weight.to_numpy(float);cap=replay.current_beta_cap.to_numpy(float);tw=target[['soxl_weight','soxx_weight','cash_weight']].to_numpy(float);risk=tw[:,0]+tw[:,1];beta=3*tw[:,0]+tw[:,1];ceiling=np.minimum.reduce([rawrisk,cap,np.full(944,.85)])
execute=replay.execute_open_target.to_numpy(bool);eps=1e-9
checks={'simplex':np.abs(tw.sum(axis=1)-1)>eps,'cash_floor':tw[:,2]<.15-eps,'beta_cap':beta>cap+eps,'risky_dollar_ceiling':risk>ceiling+eps,'negative_weight':np.min(tw,axis=1)<-eps}
report={'decision':'saved_target_semantics_diagnostic_only','source_recorder_id':src['stage1_recorder_id'],'target_rows':944,'target_columns':list(target.columns),'replay_columns':list(replay.columns),'execute_sessions':int(execute.sum()),'max_abs_simplex_error':float(np.max(np.abs(tw.sum(axis=1)-1))),'min_cash':float(np.min(tw[:,2])),'max_beta_minus_cap':float(np.max(beta-cap)),'min_ceiling_minus_target_risk':float(np.min(ceiling-risk)),'checks':{},'source_reads':2,'SOXX_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0}
for name,mask in checks.items():
    ids=np.flatnonzero(mask);report['checks'][name]={'failed_rows':len(ids),'failed_EXECUTE_rows':int(np.sum(execute[ids])),'failed_HOLD_rows':int(np.sum(~execute[ids])),'dates':pd.to_datetime(target.signal_date.iloc[ids]).dt.strftime('%Y-%m-%d').tolist(),'examples':[{'signal_date':str(pd.Timestamp(target.signal_date.iloc[i]).date()),'execute':bool(execute[i]),'raw_risk':float(rawrisk[i]),'cap':float(cap[i]),'target_soxl':float(tw[i,0]),'target_soxx':float(tw[i,1]),'target_cash':float(tw[i,2]),'target_beta':float(beta[i]),'ceiling':float(ceiling[i])} for i in ids[:5]]}
(card/'semantic_diagnostic_result.json').write_text(json.dumps(report,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({k:{a:v for a,v in row.items() if a!='dates' and a!='examples'} for k,row in report['checks'].items()}))


{"simplex": {"failed_rows": 0, "failed_EXECUTE_rows": 0, "failed_HOLD_rows": 0}, "cash_floor": {"failed_rows": 0, "failed_EXECUTE_rows": 0, "failed_HOLD_rows": 0}, "beta_cap": {"failed_rows": 0, "failed_EXECUTE_rows": 0, "failed_HOLD_rows": 0}, "risky_dollar_ceiling": {"failed_rows": 18, "failed_EXECUTE_rows": 0, "failed_HOLD_rows": 18}, "negative_weight": {"failed_rows": 0, "failed_EXECUTE_rows": 0, "failed_HOLD_rows": 0}}


逐字段诊断：目标权重闭合、现金底线和beta上限全部通过；只有18个HOLD日超过原始分配美元上界，EXECUTE零条失败。HOLD保存原持仓。
